# 01 — Chuẩn bị bộ dữ liệu và nhãn

**Quy tắc nhãn:** `1 = tương tự`, `0 = không tương tự`

In [3]:
from pathlib import Path
import sys
import pandas as pd

# Xác định thư mục gốc lab_4
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name != "lab_4":
    PROJECT_ROOT = PROJECT_ROOT / "lab_4"

# Đường dẫn đến src/dataset.py
SRC_DIR = PROJECT_ROOT / "src"
DATASET_FILE = SRC_DIR / "dataset.py"

# Kiểm tra đúng vị trí file
print("Project:", PROJECT_ROOT)
print("Dataset:", DATASET_FILE)

# Import dataset
sys.path.insert(0, str(SRC_DIR))
import dataset

Project: c:\Users\LENOVO\OneDrive\Tài liệu\học\computer vision\lab_4
Dataset: c:\Users\LENOVO\OneDrive\Tài liệu\học\computer vision\lab_4\src\dataset.py


## 1. Kiểm tra ảnh đầu vào

Cell tiếp theo kiểm tra khả năng đọc ảnh, ghi lại kích thước/định dạng, phát hiện các tệp giống hệt nhau theo SHA-256 và lưu báo cáo kiểm kê ảnh.

In [4]:
valid_images, invalid_images = dataset.inspect_images()
inventory = pd.read_csv(dataset.INVENTORY_CSV)
display(inventory)
if invalid_images:
    display(pd.DataFrame(invalid_images))

Tệp ảnh có định dạng hỗ trợ: 15
Ảnh đọc được: 15 | Ảnh lỗi: 0
Đã lưu danh sách ảnh: data\output\image_inventory.csv
Không phát hiện tệp trùng khớp byte.


,filename,path,width,height,mode,sha256
0,anh_dem.jpg,data/input/anh_dem.jpg,2730,4088,RGB,2dff44dc3e45b8ea28c4996018ac28fedb2a68a7d3cda6...
1,buom.jpg,data/input/buom.jpg,4455,2971,RGB,6c1ad008fac8d586b4b277ea992e0cebb94e11a23badd3...
2,chim.jpg,data/input/chim.jpg,6240,4160,RGB,3f23b3680252af93d24c7d1683c81a38b89aca7b7d2cd6...
3,cong_trinh.jpg,data/input/cong_trinh.jpg,3024,4032,RGB,141fee9a00bab158db7c66160e2b9f82953afce5778b1d...
4,da_mat.jpg,data/input/da_mat.jpg,2832,4256,RGB,ccbc6f72d12b8b82f01ff028ae5919db0a769afd796afa...
5,mat_meo.jpg,data/input/mat_meo.jpg,6016,4000,RGB,55957540ed47303b13f2e6eaefcd9eb2cf3d77dcb048ab...
6,meo_ngu.jpg,data/input/meo_ngu.jpg,5184,3456,RGB,a9f957d9ba3a6f9b924a22c22268a19bec8e30e492ff60...
7,nui_tuyet.jpg,data/input/nui_tuyet.jpg,3648,5472,RGB,d74e355af06cbd5a153fc150b8a79b4c239c0f0097b093...
8,pho_co.jpg,data/input/pho_co.jpg,4016,6016,RGB,be930cacf098063873aa4cd25ba34608b579ecfe62eec4...
9,puppy.jpg,data/input/puppy.jpg,4092,3163,RGB,cb8f7bc2f72ddb7cfbe8ecdf302588fac3a28cd9066500...


## 2. Tạo danh sách các cặp ảnh

Mỗi cặp không thứ tự chỉ xuất hiện một lần. Với N ảnh hợp lệ, số cặp tối đa là N×(N−1)/2. Khi chạy lại, script giữ nguyên nhãn và ghi chú đã nhập cho các cặp còn tồn tại.

In [5]:
pair_rows = dataset.create_pair_template(valid_images)
pairs = pd.read_csv(dataset.PAIRS_CSV, dtype={"label": "string"})
print("Số cặp:", len(pairs))
display(pairs.head(105))

Đã tạo/cập nhật 105 cặp: data\output\pairs.csv
Điền label=1 (tương tự) hoặc label=0 (không tương tự).
Nhãn và ghi chú cũ được giữ lại khi chạy lại chương trình.
Số cặp: 105


,image1,image2,label,notes
0,data/input/anh_dem.jpg,data/input/buom.jpg,0,Khác nhóm nội dung/chủ thể chính.
1,data/input/anh_dem.jpg,data/input/chim.jpg,0,Khác nhóm nội dung/chủ thể chính.
2,data/input/anh_dem.jpg,data/input/cong_trinh.jpg,1,Cùng nhóm cảnh đô thị/đường phố/công trình.
3,data/input/anh_dem.jpg,data/input/da_mat.jpg,0,Khác nhóm nội dung/chủ thể chính.
4,data/input/anh_dem.jpg,data/input/mat_meo.jpg,0,Khác nhóm nội dung/chủ thể chính.
...,...,...,...,...
100,data/input/rung_cay.jpg,data/input/suong_mu.jpg,0,Khác nhóm nội dung/chủ thể chính.
101,data/input/rung_cay.jpg,data/input/thu_vien.jpg,0,Khác nhóm nội dung/chủ thể chính.
102,data/input/song_bien.jpg,data/input/suong_mu.jpg,0,Khác nhóm nội dung/chủ thể chính.
103,data/input/song_bien.jpg,data/input/thu_vien.jpg,0,Khác nhóm nội dung/chủ thể chính.


## 3. Gán nhãn thực tế

Mở `data/output/pairs.csv` và điền cột `label`:

- `1`: hai ảnh tương tự theo quy tắc của nhóm.
- `0`: hai ảnh không tương tự.
- `notes`: có thể ghi lý do hoặc đặc điểm hỗ trợ quyết định.

Cần bao gồm cả cặp tương tự và không tương tự. Nếu yêu cầu là cùng một đối tượng qua góc chụp/kích thước/nhiễu, bộ ảnh phải thực sự chứa những biến thể đó. Không thể tạo các trường hợp này chỉ bằng cách đổi nhãn; cần bổ sung ảnh phù hợp vào `data/input/`.

Vì nhãn là dữ liệu nền để đánh giá, không tự điền nhãn suy đoán từ tên file hay danh mục ảnh.

## 4. Kiểm tra nhãn trước khi đánh giá

Chạy cell dưới đây sau khi đã điền xong mọi nhãn trong `pairs.csv`. Cell này phát hiện nhãn trống, nhãn sai định dạng và phân bố hai lớp.

In [6]:
label_report = dataset.validate_labels()
labels = pd.read_csv(dataset.PAIRS_CSV, dtype={"label": "string"})
display(labels["label"].fillna("(trống)").value_counts().rename_axis("label").to_frame("count"))
assert label_report["missing"] == 0, "Còn cặp chưa gán nhãn."
assert label_report["invalid"] == 0, "Có nhãn khác 0/1."
assert label_report["positive"] > 0 and label_report["negative"] > 0, "Cần có cả nhãn 0 và nhãn 1." 

Tổng số cặp: 105
Nhãn 1: 19 | Nhãn 0: 86
Cặp còn thiếu nhãn: 0 | Nhãn không hợp lệ: 0
Nhãn đã hợp lệ về định dạng; tiếp theo cần kiểm tra tính nhất quán của nhãn.


,count
label,
0,86
1,19


## Sản phẩm bàn giao

- `src/dataset.py`: mã kiểm tra dữ liệu, kiểm kê ảnh, tạo cặp và xác thực nhãn.
- `notebook/01_dataset_preparation.ipynb`: các bước thực hiện và xem kết quả.
- `data/input/`: ảnh gốc do nhóm chuẩn bị.
- `data/output/image_inventory.csv`: metadata ảnh, trạng thái đọc và mã SHA-256.
- `data/output/pairs.csv`: các cặp ảnh, nhãn thực tế và ghi chú.

**Lưu ý:** Accuracy, Recall và ROC chỉ có ý nghĩa sau khi nhãn đã được xác nhận và mô hình đã tạo dự đoán cho các cặp ảnh.